# **Data Preparation:**

**`Problem`:**

   1.  **`Identify the main drivers of late deliveries`**

   2.  **`Understand what causes low customer ratings`** 

   3.  **`Determine if specific warehouse blocks are underperforming`** 

   4.  **`Optimize the mode of shipment`** 

   5.  **`Assess the impact of discounts`** 

---------

In [2]:
import numpy as np 
import pandas as pd

In [3]:
data= pd.read_csv("https://raw.githubusercontent.com/rajesh-coventry/Projects/refs/heads/master/00_DataSets/31_E-Commerce%20Shipping%20Data.csv")
data.columns

Index(['ID', 'Warehouse_block', 'Mode_of_Shipment', 'Customer_care_calls',
       'Customer_rating', 'Cost_of_the_Product', 'Prior_purchases',
       'Product_importance', 'Gender', 'Discount_offered', 'Weight_in_gms',
       'Reached.on.Time_Y.N'],
      dtype='object')

In [4]:
data.head(3)

,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
0,1,D,Flight,4,2,177,3,low,F,44,1233,1
1,2,F,Flight,4,5,216,2,low,M,59,3088,1
2,3,A,Flight,2,2,183,4,low,M,48,3374,1


**`ID`** - Unique identifier for each shipping record

**`Warehouse_block`** - Warehouse location/block where product is stored

**`Mode_of_Shipment`** - Shipping method used (e.g., air, sea, road)

**`Customer_care_calls`** - Number of customer service calls made

**`Customer_rating`** - Customer satisfaction rating for the product/service

**`Cost_of_the_Product`** - Price of the shipped product

**`Prior_purchases`** - Number of previous purchases by the customer

**`Product_importance`** - Priority level of the product (low, medium, high)

**`Gender`** - Customer's gender

**`Discount_offered`** - Percentage discount given on the product

**`Weight_in_gms`** - Product weight in grams

**`Reached.on.Time_Y.N`** - Target variable indicating if delivery was on time (Yes/No)

This is a binary classification dataset for `predicting delivery timeliness` based on various `shipping and customer factors`.

In [5]:
# checking for any missing values:  
data.isnull().any()

ID                     False
Warehouse_block        False
Mode_of_Shipment       False
Customer_care_calls    False
Customer_rating        False
Cost_of_the_Product    False
Prior_purchases        False
Product_importance     False
Gender                 False
Discount_offered       False
Weight_in_gms          False
Reached.on.Time_Y.N    False
dtype: bool

In [6]:
data.isnull().any().sum()

np.int64(0)

So, no missing or null values in the data. 

In [7]:
# Checking for duplicate rows in the dataset: 
data.duplicated().any()

np.False_

So, dataset also contains no duplicated rows.

In [8]:
# Data types of different columns: 
data.dtypes

ID                      int64
Warehouse_block        object
Mode_of_Shipment       object
Customer_care_calls     int64
Customer_rating         int64
Cost_of_the_Product     int64
Prior_purchases         int64
Product_importance     object
Gender                 object
Discount_offered        int64
Weight_in_gms           int64
Reached.on.Time_Y.N     int64
dtype: object

In [9]:
data.sample(3)

,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
1395,1396,B,Ship,4,2,158,4,low,F,45,1758,1
5516,5517,A,Ship,6,2,290,4,medium,M,2,1070,1
1713,1714,B,Ship,4,2,200,3,medium,M,45,2486,1


In [10]:
# make all the column names shorter and lower case for ease of typing: 
data= data.rename(columns= {"ID": "id", "Warehouse_block": "warehouse_block", 
                            "Mode_of_Shipment": "mode_of_shipment", 
                             "Customer_care_calls": "customer_care_calls", 
                              "Customer_rating": "customer_rating", "Cost_of_the_Product":
                               "cost_of_product", "Prior_purchases": "prior_purchase", 
                                "Product_importance": "product_importance", "Gender": "gender", 
                                 "Discount_offered": "discount_offered", "Weight_in_gms": "weight_gms", 
                                  "Reached.on.Time_Y.N": "reached_on_time" })

In [11]:
data.columns

Index(['id', 'warehouse_block', 'mode_of_shipment', 'customer_care_calls',
       'customer_rating', 'cost_of_product', 'prior_purchase',
       'product_importance', 'gender', 'discount_offered', 'weight_gms',
       'reached_on_time'],
      dtype='object')

In [14]:
# Check the correlation between different variables with the target: reached_on_time: 
data.corr(numeric_only= True)["reached_on_time"]

id                    -0.411822
customer_care_calls   -0.067126
customer_rating        0.013119
cost_of_product       -0.073587
prior_purchase        -0.055515
discount_offered       0.397108
weight_gms            -0.268793
reached_on_time        1.000000
Name: reached_on_time, dtype: float64

We do not consider `id` here because, it has nothing to do with if the product was dlivered on time or not. 

Also, correlation of `reached_on_time` with itself is 1 which is obvious because the self-correlation of any feature is always maximum. 

Other then that it appears that, there is no positive or negative strong correlation between the target variable with other variables. `discount_offered` have moderate positive correlation: considering all other variables constant, there is a moderate linear relationship between `reached_on_time` with `discount_offered`. This means, if `discount_offered` tends to increase then, the `delivery_on_time` also increases linearly at moderate rate. Correlation always  gives the linear relationship between variables and if there is a high degree of non-linear relation between the variables then, the correlation coefficient do not explain that part of the relation.

All other variables have negative low correlation values indicating there is almost no linear relation (or, have very low degree of negative association) between them. 

The variables may exhibit non-linear relationship with `delivery_on_time` and need much advanced techniques to analyze and solve the problem. 

In [15]:
data.columns

Index(['id', 'warehouse_block', 'mode_of_shipment', 'customer_care_calls',
       'customer_rating', 'cost_of_product', 'prior_purchase',
       'product_importance', 'gender', 'discount_offered', 'weight_gms',
       'reached_on_time'],
      dtype='object')

In [18]:
data.shape

(10999, 12)

In [17]:
data["id"].nunique()

10999

In [19]:
data["warehouse_block"].unique()

array(['D', 'F', 'A', 'B', 'C'], dtype=object)

In [20]:
data["mode_of_shipment"].unique()

array(['Flight', 'Ship', 'Road'], dtype=object)

In [21]:
data["customer_care_calls"].unique()

array([4, 2, 3, 5, 6, 7])

In [22]:
data["customer_rating"].unique()

array([2, 5, 3, 1, 4])

In [26]:
data["cost_of_product"].dtype

dtype('int64')

In [27]:
data["prior_purchase"].unique()

array([ 3,  2,  4,  6,  5,  7, 10,  8])

In [28]:
data["product_importance"].unique()

array(['low', 'medium', 'high'], dtype=object)

In [29]:
data["gender"].unique()

array(['F', 'M'], dtype=object)

In [30]:
data["discount_offered"].unique()

array([44, 59, 48, 10, 46, 12,  3, 11, 29, 32,  1, 43, 45,  6, 36, 18, 38,
       51,  2, 28, 24, 31, 61, 22,  4, 62, 16, 56, 15,  9, 40, 37, 41, 17,
       64, 52, 49, 39, 14, 33, 21, 13, 23, 26, 57,  7, 35,  8,  5, 53, 55,
       47, 65, 25, 50, 60, 20, 19, 63, 58, 34, 54, 27, 42, 30])

In [31]:
data["weight_gms"].dtype

dtype('int64')

In [32]:
data['reached_on_time'].unique()

array([1, 0])

> > `0` ---> Indicates Not Reached on Time 

> > `1` ---> Indicates Reached on Time

In [33]:
# store data in json format: 
data.to_json("data.json")

Check if the dataset was saved successfully or not:

In [35]:
pd.read_json("data.json").columns

Index(['id', 'warehouse_block', 'mode_of_shipment', 'customer_care_calls',
       'customer_rating', 'cost_of_product', 'prior_purchase',
       'product_importance', 'gender', 'discount_offered', 'weight_gms',
       'reached_on_time'],
      dtype='object')

Everything works fine now.